# Huấn luyện & đánh giá - từng bước

Notebook minh họa một thí nghiệm đơn lẻ (TF-IDF + Logistic Regression) để hiểu luồng xử lý; thí nghiệm đầy đủ chạy bằng `python -m src.train`.

Cần chạy trước: `python -m src.preprocess`.

In [ ]:
import os, sys
from pathlib import Path

# Đưa thư mục gốc dự án vào sys.path để `import src...` hoạt động dù notebook chạy từ thư mục nào.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import config as C

## 1. Nạp dữ liệu đã tiền xử lý

In [ ]:
from src.preprocess import load_processed

data = load_processed()
data['train'][[C.TEXT_COL, 'text_clean', 'text_seg', 'sentiment', 'topic']].head(5)

## 2. TF-IDF (n-gram 1-2) - fit trên train, biến đổi cả 3 tập

In [ ]:
from src.features import build_features

fs = build_features('tfidf', data, 'text_seg')
print('Ma trận train:', fs.X_train.shape, '| số từ vựng:', len(fs.vectorizer.vocabulary_))

## 3. Huấn luyện một mô hình (chọn C theo validation)

In [ ]:
from src.train import tune_and_evaluate

task = 'sentiment'
y = {s: data[s][task].to_numpy() for s in C.SPLITS}
res = tune_and_evaluate('LogisticRegression', fs, y['train'], y['validation'], y['test'], task, 'balanced')
pd.Series(res.row)[['params', 'val_f1_macro', 'test_accuracy', 'test_f1_macro', 'test_f1_weighted']]

## 4. Báo cáo phân loại & ma trận nhầm lẫn (tập test)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
from src.evaluate import plot_confusion_matrix

y_pred = res.estimator.predict(fs.X_test)
print(classification_report(y['test'], y_pred, target_names=C.LABEL_VI[task], digits=4, zero_division=0))

out = C.FIGURES_DIR / 'notebook_cm.png'
plot_confusion_matrix(np.array(res.cm), C.LABEL_VI[task], 'Ma trận nhầm lẫn (notebook)', out)
display(Image(filename=str(out), width=800))

## 5. Chạy toàn bộ lưới thí nghiệm
Tốn vài phút đến vài chục phút. Bỏ dấu `#` để chạy (hoặc dùng Terminal: `python -m src.train --ablation`).

In [ ]:
# from src.train import run_experiments
# results = run_experiments(['bow', 'tfidf'], list(C.MODEL_NAMES), list(C.TASKS), 'balanced', C.EMBEDDING_MODEL)
# results.sort_values('test_f1_macro', ascending=False).head(10)

## 6. Dự đoán thử bằng mô hình đã lưu
(Cần đã chạy `python -m src.train` để có file trong `models/`.)

In [ ]:
from src.predict import FeedbackPredictor

predictor = FeedbackPredictor()
for s in ['Thầy dạy rất dễ hiểu và nhiệt tình', 'Phòng học nóng và wifi yếu quá', 'Môn học bình thường']:
    r = predictor.analyze(s)
    print(f"{s!r:<45} -> {r['sentiment']['label_vi']} ({r['sentiment']['confidence']:.0%}) | {r['topic']['label_vi']} ({r['topic']['confidence']:.0%})")